# Database Schema Viewer
This notebook connects to the SQL Server database using the credentials from the `.env` file and displays the database schema.

In [2]:
import os
import urllib.parse
import sqlalchemy as sa
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase

# Load environment variables from .env file
load_dotenv()

True

### Connect to Database
Here we construct the connection string and test the connection.

In [3]:
DB_SERVER = os.getenv("DB_SERVER", "localhost")
DB_PORT   = os.getenv("DB_PORT", "1433")
DB_USER   = os.getenv("DB_USER", "")
DB_PASS   = os.getenv("DB_PASS", "")
DB_NAME   = os.getenv("DB_NAME", "NexonDB")

if DB_USER and DB_PASS:
    connection_string = (
        f"mssql+pyodbc://{DB_USER}:{urllib.parse.quote_plus(DB_PASS)}"
        f"@{DB_SERVER}:{DB_PORT}/{DB_NAME}"
        "?driver=ODBC+Driver+17+for+SQL+Server"
        "&Encrypt=no&TrustServerCertificate=YES"
    )
else:
    connection_string = (
        f"mssql+pyodbc://@{DB_SERVER}/{DB_NAME}"
        "?driver=ODBC+Driver+17+for+SQL+Server"
        "&Trusted_Connection=yes&TrustServerCertificate=YES"
    )

try:
    engine = sa.create_engine(connection_string)
    # Test connection
    with engine.connect() as conn:
        print("Successfully connected to the database!")
except Exception as e:
    print(f"Failed to connect: {e}")

C:\Users\Asus\AppData\Local\Temp\ipykernel_26288\3413723375.py:24: SAWarning: Unrecognized server version info '17.0.4025.3'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


Successfully connected to the database!


### Display Schema
Using Langchain's SQLDatabase utility to fetch and display the schema.

In [4]:
try:
    db = SQLDatabase.from_uri(connection_string)
    table_info = db.table_info
    
    print("="*50)
    print("DATABASE SCHEMA")
    print("="*50)
    print(table_info)
except Exception as e:
    print(f"Failed to fetch schema: {e}")

d:\anaconda\Lib\site-packages\langchain_community\utilities\sql_database.py:81: SAWarning: Unrecognized server version info '17.0.4025.3'.  Some SQL Server features may not function properly.
  self._inspector = inspect(self._engine)


DATABASE SCHEMA

CREATE TABLE [AspNetRoleClaims] (
	[Id] INTEGER NOT NULL IDENTITY(1,1), 
	[RoleId] NVARCHAR(450) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL, 
	[ClaimType] NVARCHAR(max) COLLATE SQL_Latin1_General_CP1_CI_AS NULL, 
	[ClaimValue] NVARCHAR(max) COLLATE SQL_Latin1_General_CP1_CI_AS NULL, 
	CONSTRAINT [PK_AspNetRoleClaims] PRIMARY KEY ([Id]), 
	CONSTRAINT [FK_AspNetRoleClaims_AspNetRoles_RoleId] FOREIGN KEY([RoleId]) REFERENCES [AspNetRoles] ([Id]) ON DELETE CASCADE
)

/*
3 rows from AspNetRoleClaims table:
Id	RoleId	ClaimType	ClaimValue

*/


CREATE TABLE [AspNetRoles] (
	[Id] NVARCHAR(450) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL, 
	[Name] NVARCHAR(256) COLLATE SQL_Latin1_General_CP1_CI_AS NULL, 
	[NormalizedName] NVARCHAR(256) COLLATE SQL_Latin1_General_CP1_CI_AS NULL, 
	[ConcurrencyStamp] NVARCHAR(max) COLLATE SQL_Latin1_General_CP1_CI_AS NULL, 
	CONSTRAINT [PK_AspNetRoles] PRIMARY KEY ([Id])
)

/*
3 rows from AspNetRoles table:
Id	Name	NormalizedName	Concurrency

In [8]:
import sqlalchemy as sa

try:
    # إنشاء Inspector لقاعدة البيانات
    inspector = sa.inspect(engine)
    
    # جلب جميع أسماء الجداول
    tables = inspector.get_table_names()
    
    print("=" * 50)
    print(f"Total Tables Found: {len(tables)}")
    print("=" * 50)
    
    for table in tables:
        print(f"- {table}")
        
except Exception as e:
    print(f"Failed to fetch tables: {e}")


Total Tables Found: 23
- __EFMigrationsHistory
- AspNetRoleClaims
- AspNetRoles
- AspNetUserClaims
- AspNetUserLogins
- AspNetUserRoles
- AspNetUsers
- AspNetUserTokens
- Categories
- ChatMessages
- Favorites
- PlatformSettings
- ProductImages
- ProductReviews
- Products
- RentalOrderDetails
- RentalOrders
- Reviews
- Subcategories
- Subscriptions
- UserAddresses
- UserSubscriptions
- WalletTransactions


In [9]:
import pandas as pd

try:
    # كتابة الاستعلام لجلب كل البيانات من جدول Products
    query = "SELECT * FROM Products"
    
    # قراءة البيانات وعرضها في DataFrame
    df_products = pd.read_sql_query(query, engine)
    
    # عرض البيانات (ستظهر كجدول منسق)
    display(df_products)
    
    # إذا كان الجدول كبيراً جداً وتريد عرض أول 10 صفوف فقط، يمكنك استخدام:
    # display(df_products.head(10))
    
except Exception as e:
    print(f"Failed to fetch data: {e}")


,Id,UserId,CategoryId,SubcategoryId,LocationArea,Condition,ProductType,Brand,RentalGuarantee,Name,Description,FinalPricePerDay,TermsConditions,Status,CreatedAt,AverageRating,BasePricePerDay,TotalReviews,TotalPlatformProfit,TotalRentalCount
0,5,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,1,101,Nasr City,1,Electronics,Sony,1,PlayStation 5,PS5 with 2 controllers,200.0,Return in same condition,2,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
1,6,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,1,102,Heliopolis,2,Electronics,Microsoft,1,Xbox Series X,Xbox with wireless controller,180.0,No damage allowed,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
2,7,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,2,201,Maadi,1,Sports,Trinx,0,Mountain Bike,Professional mountain bike,90.0,Helmet included,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
3,8,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,2,202,Dokki,2,Sports,Galaxy,0,City Bike,Comfortable city bicycle,70.0,Return clean,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
4,9,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,3,301,Nasr City,1,Electronics,Canon,1,Canon DSLR Camera,Canon DSLR with lens,150.0,No scratches,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
5,10,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,3,302,Helwan,2,Electronics,Nikon,1,Nikon Camera,Nikon camera with tripod,140.0,Handle carefully,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
6,11,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,4,401,Zamalek,1,Computers,Dell,1,Dell Laptop,Core i7 laptop 16GB RAM,220.0,No software install,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
7,12,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,4,402,Nasr City,2,Computers,HP,0,HP Laptop,Core i5 laptop,150.0,Return charged,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
8,13,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,5,501,Maadi,1,Projectors,Epson,1,Epson Projector,Full HD projector,130.0,Indoor use only,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
9,14,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,5,502,Dokki,2,Projectors,BenQ,1,BenQ Projector,Portable projector,120.0,No dropping,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0


In [10]:
from sqlalchemy import text

try:
    # فتح اتصال بقاعدة البيانات
    with engine.connect() as conn:
        # جلب أول 10 منتجات (ويمكنك إزالة TOP 10 لجلب الكل)
        result = conn.execute(text("SELECT TOP 10 * FROM Products"))
        
        # طباعة أسماء الأعمدة (Columns)
        print("Columns:", list(result.keys()))
        print("-" * 80)
        
        # طباعة الصفوف
        for row in result:
            print(row)
            
except Exception as e:
    print(f"Failed to fetch data: {e}")


Columns: ['Id', 'UserId', 'CategoryId', 'SubcategoryId', 'LocationArea', 'Condition', 'ProductType', 'Brand', 'RentalGuarantee', 'Name', 'Description', 'FinalPricePerDay', 'TermsConditions', 'Status', 'CreatedAt', 'AverageRating', 'BasePricePerDay', 'TotalReviews', 'TotalPlatformProfit', 'TotalRentalCount']
--------------------------------------------------------------------------------
(5, '8b0c078d-f802-4b7d-a8db-88cf2410a6cf', 1, 101, 'Nasr City', 1, 'Electronics', 'Sony', '1', 'PlayStation 5', 'PS5 with 2 controllers', Decimal('200.00'), 'Return in same condition', 2, datetime.datetime(2026, 4, 7, 14, 47, 41, 552038), 0.0, Decimal('0.00'), 0, Decimal('0.00'), 0)
(6, '8b0c078d-f802-4b7d-a8db-88cf2410a6cf', 1, 102, 'Heliopolis', 2, 'Electronics', 'Microsoft', '1', 'Xbox Series X', 'Xbox with wireless controller', Decimal('180.00'), 'No damage allowed', 1, datetime.datetime(2026, 4, 7, 14, 47, 41, 552038), 0.0, Decimal('0.00'), 0, Decimal('0.00'), 0)
(7, '8b0c078d-f802-4b7d-a8db-88cf2

In [11]:
import pandas as pd
import random
from datetime import datetime

try:
    # 1. جلب قائمة بـ IDs المستخدمين الحقيقيين من قاعدة البيانات
    users_df = pd.read_sql_query("SELECT Id FROM AspNetUsers", engine)
    valid_user_ids = users_df['Id'].tolist()
    
    if not valid_user_ids:
        print("⚠️ جدول المستخدمين AspNetUsers فارغ! لا يمكنك إضافة منتجات قبل تسجيل حساب مستخدم واحد على الأقل في النظام.")
        
    else:
        print(f"تم العثور على {len(valid_user_ids)} مستخدمين حقيقيين في النظام. سيتم التوزيع عليهم عشوائياً...")
        
        # 2. تجهيز البيانات لكن المرة دي هنسحب UserId عشوائي من اللي موجودين في القاعدة
        # وكمان حولنا النصوص (New/Available) لأرقام مباشرة (1 و 2)
        raw_data = [
            [random.choice(valid_user_ids),1,101,"Nasr City",1,"Electronics","Sony",1,"PlayStation 5", "PS5 with 2 controllers",200,"Return in same condition",1,datetime.now()],
            [random.choice(valid_user_ids),1,102,"Heliopolis",2,"Electronics","Microsoft",1,"Xbox Series X", "Xbox with wireless controller",180,"No damage allowed",1,datetime.now()],
            [random.choice(valid_user_ids),2,201,"Maadi",1,"Sports","Trinx",0,"Mountain Bike", "Professional mountain bike",90,"Helmet included",1,datetime.now()],
            [random.choice(valid_user_ids),2,202,"Dokki",2,"Sports","Galaxy",0,"City Bike", "Comfortable city bicycle",70,"Return clean",1,datetime.now()],
            [random.choice(valid_user_ids),3,301,"Nasr City",1,"Electronics","Canon",1,"Canon DSLR Camera", "Canon DSLR with lens",150,"No scratches",1,datetime.now()],
            [random.choice(valid_user_ids),3,302,"Helwan",2,"Electronics","Nikon",1,"Nikon Camera", "Nikon camera with tripod",140,"Handle carefully",1,datetime.now()],
            [random.choice(valid_user_ids),4,401,"Zamalek",1,"Computers","Dell",1,"Dell Laptop", "Core i7 laptop 16GB RAM",220,"No software install",1,datetime.now()],
            [random.choice(valid_user_ids),4,402,"Nasr City",2,"Computers","HP",0,"HP Laptop", "Core i5 laptop",150,"Return charged",1,datetime.now()],
            [random.choice(valid_user_ids),5,501,"Maadi",1,"Projectors","Epson",1,"Epson Projector", "Full HD projector",130,"Indoor use only",1,datetime.now()],
            [random.choice(valid_user_ids),5,502,"Dokki",2,"Projectors","BenQ",1,"BenQ Projector", "Portable projector",120,"No dropping",1,datetime.now()],
            [random.choice(valid_user_ids),6,601,"Heliopolis",1,"Tools","Bosch",0,"Electric Drill", "Bosch heavy-duty drill",60,"Return with case",1,datetime.now()],
            [random.choice(valid_user_ids),6,602,"Nasr City",2,"Tools","Makita",0,"Angle Grinder", "Makita grinder",55,"Safety required",1,datetime.now()],
            [random.choice(valid_user_ids),7,701,"Maadi",1,"Gaming","Logitech",0,"Gaming Steering Wheel", "Wheel for racing games",80,"Return intact",1,datetime.now()],
            [random.choice(valid_user_ids),7,702,"Dokki",2,"Gaming","Razer",0,"Gaming Keyboard", "RGB mechanical keyboard",40,"No liquid damage",1,datetime.now()],
            [random.choice(valid_user_ids),8,801,"Zamalek",1,"Audio","JBL",0,"JBL Speaker", "Portable Bluetooth speaker",50,"Return charged",1,datetime.now()],
            [random.choice(valid_user_ids),8,802,"Nasr City",2,"Audio","Sony",0,"Sony Headphones", "Noise cancelling headphones",45,"No scratches",1,datetime.now()],
            [random.choice(valid_user_ids),9,901,"Heliopolis",1,"Photography","GoPro",1,"GoPro Hero 11", "Action camera waterproof",160,"Return accessories",1,datetime.now()],
            [random.choice(valid_user_ids),9,902,"Maadi",2,"Photography","DJI",1,"DJI Gimbal", "Camera stabilizer",100,"Handle carefully",1,datetime.now()],
            [random.choice(valid_user_ids),10,1001,"Nasr City",1,"Furniture","IKEA",0,"Office Chair", "Comfortable office chair",30,"Indoor use only",1,datetime.now()],
            [random.choice(valid_user_ids),10,1002,"Dokki",2,"Furniture","Generic",0,"Folding Table", "Portable folding table",25,"Return clean",1,datetime.now()]
        ]

        products_data = pd.DataFrame(raw_data, columns=[
            "UserId","CategoryId","SubcategoryId","LocationArea","Condition",
            "ProductType","Brand","RentalGuarantee","Name","Description",
            "PricePerDay","TermsConditions","Status","CreatedAt"
        ])

        # 3. حفظ البيانات في الـ Database
        products_data.to_sql('Products', con=engine, if_exists='append', index=False)
        print("تم إضافة الـ 20 منتج بنجاح ✅")

except Exception as e:
    print(f"حدث خطأ أثناء الإضافة: {e}")

تم العثور على 14 مستخدمين حقيقيين في النظام. سيتم التوزيع عليهم عشوائياً...
حدث خطأ أثناء الإضافة: (pyodbc.ProgrammingError) ('42S22', "[42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'PricePerDay'. (207) (SQLExecDirectW); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'PricePerDay'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")
[SQL: INSERT INTO [Products] ([UserId], [CategoryId], [SubcategoryId], [LocationArea], [Condition], [ProductType], [Brand], [RentalGuarantee], [Name], [Description], [PricePerDay], [TermsConditions], [Status], [CreatedAt]) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ? ... 752 characters truncated ...  ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('eb43366e-0762-4f8a-9a43-ea0e0d542705', 1, 101, 'Nasr City', 1, 'Electronics', 'Sony', 1, 'PlayStation 5', 'PS5 with 2 control

In [12]:
import pandas as pd

try:
    # كتابة الاستعلام لجلب كل البيانات من جدول Products
    query = "SELECT * FROM Products"
    
    # قراءة البيانات وعرضها في DataFrame
    df_products = pd.read_sql_query(query, engine)
    
    # عرض البيانات (ستظهر كجدول منسق)
    display(df_products)
    
    # إذا كان الجدول كبيراً جداً وتريد عرض أول 10 صفوف فقط، يمكنك استخدام:
    # display(df_products.head(10))
    
except Exception as e:
    print(f"Failed to fetch data: {e}")

,Id,UserId,CategoryId,SubcategoryId,LocationArea,Condition,ProductType,Brand,RentalGuarantee,Name,Description,FinalPricePerDay,TermsConditions,Status,CreatedAt,AverageRating,BasePricePerDay,TotalReviews,TotalPlatformProfit,TotalRentalCount
0,5,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,1,101,Nasr City,1,Electronics,Sony,1,PlayStation 5,PS5 with 2 controllers,200.0,Return in same condition,2,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
1,6,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,1,102,Heliopolis,2,Electronics,Microsoft,1,Xbox Series X,Xbox with wireless controller,180.0,No damage allowed,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
2,7,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,2,201,Maadi,1,Sports,Trinx,0,Mountain Bike,Professional mountain bike,90.0,Helmet included,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
3,8,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,2,202,Dokki,2,Sports,Galaxy,0,City Bike,Comfortable city bicycle,70.0,Return clean,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
4,9,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,3,301,Nasr City,1,Electronics,Canon,1,Canon DSLR Camera,Canon DSLR with lens,150.0,No scratches,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
5,10,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,3,302,Helwan,2,Electronics,Nikon,1,Nikon Camera,Nikon camera with tripod,140.0,Handle carefully,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
6,11,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,4,401,Zamalek,1,Computers,Dell,1,Dell Laptop,Core i7 laptop 16GB RAM,220.0,No software install,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
7,12,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,4,402,Nasr City,2,Computers,HP,0,HP Laptop,Core i5 laptop,150.0,Return charged,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
8,13,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,5,501,Maadi,1,Projectors,Epson,1,Epson Projector,Full HD projector,130.0,Indoor use only,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0
9,14,8b0c078d-f802-4b7d-a8db-88cf2410a6cf,5,502,Dokki,2,Projectors,BenQ,1,BenQ Projector,Portable projector,120.0,No dropping,1,2026-04-07 14:47:41.552038,0.0,0.0,0,0.0,0


In [13]:
import pandas as pd
import uuid
import random
from datetime import datetime

# 1. إنشاء حساب مستخدم وهمي (تجريبي)
user_id = str(uuid.uuid4()) # إنشاء ID فريد للمستخدم
national_id = str(random.randint(10000000000000, 99999999999999)) # رقم قومي عشوائي من 14 رقم
username = f"testuser_{random.randint(1000,9999)}"

new_user = pd.DataFrame([{
    "Id": user_id,
    "FullName": "Test User",
    "NationalId": national_id,
    "IdCardImage": "dummy_image.jpg",
    "AccountStatus": 1,
    "Balance": 0.0,
    "CreatedAt": datetime.now(),
    "UserName": username,
    "NormalizedUserName": username.upper(),
    "Email": f"{username}@test.com",
    "NormalizedEmail": f"{username.upper()}@TEST.COM",
    "EmailConfirmed": False,
    "PhoneNumberConfirmed": False,
    "TwoFactorEnabled": False,
    "LockoutEnabled": False,
    "AccessFailedCount": 0
}])

try:
    # 2. حفظ المستخدم في قاعدة البيانات
    new_user.to_sql('AspNetUsers', con=engine, if_exists='append', index=False)
    print(f"تم إنشاء حساب مستخدم بنجاح! الـ ID الخاص به هو: {user_id}")
    
    # 3. إعداد بيانات المنتجات وربطها بحساب المستخدم الجديد
    raw_data = [
        [user_id,1,101,"Nasr City",1,"Electronics","Sony",1,"PlayStation 5", "PS5 with 2 controllers",200,"Return in same condition",1,datetime.now()],
        [user_id,1,102,"Heliopolis",2,"Electronics","Microsoft",1,"Xbox Series X", "Xbox with wireless controller",180,"No damage allowed",1,datetime.now()],
        [user_id,2,201,"Maadi",1,"Sports","Trinx",0,"Mountain Bike", "Professional mountain bike",90,"Helmet included",1,datetime.now()],
        [user_id,2,202,"Dokki",2,"Sports","Galaxy",0,"City Bike", "Comfortable city bicycle",70,"Return clean",1,datetime.now()],
        [user_id,3,301,"Nasr City",1,"Electronics","Canon",1,"Canon DSLR Camera", "Canon DSLR with lens",150,"No scratches",1,datetime.now()],
        [user_id,3,302,"Helwan",2,"Electronics","Nikon",1,"Nikon Camera", "Nikon camera with tripod",140,"Handle carefully",1,datetime.now()],
        [user_id,4,401,"Zamalek",1,"Computers","Dell",1,"Dell Laptop", "Core i7 laptop 16GB RAM",220,"No software install",1,datetime.now()],
        [user_id,4,402,"Nasr City",2,"Computers","HP",0,"HP Laptop", "Core i5 laptop",150,"Return charged",1,datetime.now()],
        [user_id,5,501,"Maadi",1,"Projectors","Epson",1,"Epson Projector", "Full HD projector",130,"Indoor use only",1,datetime.now()],
        [user_id,5,502,"Dokki",2,"Projectors","BenQ",1,"BenQ Projector", "Portable projector",120,"No dropping",1,datetime.now()],
        [user_id,6,601,"Heliopolis",1,"Tools","Bosch",0,"Electric Drill", "Bosch heavy-duty drill",60,"Return with case",1,datetime.now()],
        [user_id,6,602,"Nasr City",2,"Tools","Makita",0,"Angle Grinder", "Makita grinder",55,"Safety required",1,datetime.now()],
        [user_id,7,701,"Maadi",1,"Gaming","Logitech",0,"Gaming Steering Wheel", "Wheel for racing games",80,"Return intact",1,datetime.now()],
        [user_id,7,702,"Dokki",2,"Gaming","Razer",0,"Gaming Keyboard", "RGB mechanical keyboard",40,"No liquid damage",1,datetime.now()],
        [user_id,8,801,"Zamalek",1,"Audio","JBL",0,"JBL Speaker", "Portable Bluetooth speaker",50,"Return charged",1,datetime.now()],
        [user_id,8,802,"Nasr City",2,"Audio","Sony",0,"Sony Headphones", "Noise cancelling headphones",45,"No scratches",1,datetime.now()],
        [user_id,9,901,"Heliopolis",1,"Photography","GoPro",1,"GoPro Hero 11", "Action camera waterproof",160,"Return accessories",1,datetime.now()],
        [user_id,9,902,"Maadi",2,"Photography","DJI",1,"DJI Gimbal", "Camera stabilizer",100,"Handle carefully",1,datetime.now()],
        [user_id,10,1001,"Nasr City",1,"Furniture","IKEA",0,"Office Chair", "Comfortable office chair",30,"Indoor use only",1,datetime.now()],
        [user_id,10,1002,"Dokki",2,"Furniture","Generic",0,"Folding Table", "Portable folding table",25,"Return clean",1,datetime.now()]
    ]

    products_data = pd.DataFrame(raw_data, columns=[
        "UserId","CategoryId","SubcategoryId","LocationArea","Condition",
        "ProductType","Brand","RentalGuarantee","Name","Description",
        "PricePerDay","TermsConditions","Status","CreatedAt"
    ])

    # 4. حفظ بيانات المنتجات
    products_data.to_sql('Products', con=engine, if_exists='append', index=False)
    print("تم إضافة الـ 20 منتج بنجاح لرسلهم للمستخدم الجديد ✅")

except Exception as e:
    print(f"حدث خطأ أثناء التنفيذ: {e}")


تم إنشاء حساب مستخدم بنجاح! الـ ID الخاص به هو: 3d9aed33-6f6c-4b29-8d23-283a77257c7e
حدث خطأ أثناء التنفيذ: (pyodbc.ProgrammingError) ('42S22', "[42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'PricePerDay'. (207) (SQLExecDirectW); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'PricePerDay'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")
[SQL: INSERT INTO [Products] ([UserId], [CategoryId], [SubcategoryId], [LocationArea], [Condition], [ProductType], [Brand], [RentalGuarantee], [Name], [Description], [PricePerDay], [TermsConditions], [Status], [CreatedAt]) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ? ... 752 characters truncated ...  ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('3d9aed33-6f6c-4b29-8d23-283a77257c7e', 1, 101, 'Nasr City', 1, 'Electronics', 'Sony', 1, 'PlayStation 5', 'PS5 with 

In [14]:
import pandas as pd
import uuid
import random
from datetime import datetime
from sqlalchemy import text

try:
    with engine.begin() as conn:
        # 1. إدراج أو تجهيز الأقسام الرئيسية (Categories)
        categories = [(1, 'Electronics'), (2, 'Sports'), (3, 'Cameras'), (4, 'Computers'), 
                      (5, 'Projectors'), (6, 'Tools'), (7, 'Gaming'), (8, 'Audio'), 
                      (9, 'Photography'), (10, 'Furniture')]
        
        print("جاري إنشاء الأقسام (Categories)...")
        for cid, cname in categories:
            # نحاول إضافة الأقسام مع تفعيل الـ Identity Insert أو بدونها
            try:
                conn.execute(text(f"SET IDENTITY_INSERT Categories ON; INSERT INTO Categories (Id, Name, CreatedAt) VALUES ({cid}, '{cname}', GETDATE()); SET IDENTITY_INSERT Categories OFF;"))
            except:
                try:
                    conn.execute(text(f"INSERT INTO Categories (Id, Name, CreatedAt) VALUES ({cid}, '{cname}', GETDATE())"))
                except:
                    pass # إذا كان القسم موجود مسبقاً، نتجاهل الإضافة

        # 2. إدراج الأقسام الفرعية (SubCategories)
        subcategories = [(101, 1, 'PlayStation'), (102, 1, 'Xbox'), (201, 2, 'Bikes'), (202, 2, 'City Bikes'),
                         (301, 3, 'DSLR'), (302, 3, 'Other'), (401, 4, 'Laptops'), (402, 4, 'PCs'),
                         (501, 5, 'HD Projectors'), (502, 5, 'Portable Projectors'), (601, 6, 'Drills'), (602, 6, 'Grinders'),
                         (701, 7, 'Steering Wheels'), (702, 7, 'Keyboards'), (801, 8, 'Speakers'), (802, 8, 'Headphones'),
                         (901, 9, 'Action Cam'), (902, 9, 'Gimbals'), (1001, 10, 'Chairs'), (1002, 10, 'Tables')]
                         
        print("جاري إنشاء الأقسام الفرعية (SubCategories)...")
        for sid, cid, sname in subcategories:
            try:
                conn.execute(text(f"SET IDENTITY_INSERT SubCategories ON; INSERT INTO SubCategories (Id, CategoryId, Name) VALUES ({sid}, {cid}, '{sname}'); SET IDENTITY_INSERT SubCategories OFF;"))
            except:
                try:
                    conn.execute(text(f"INSERT INTO SubCategories (Id, CategoryId, Name) VALUES ({sid}, {cid}, '{sname}')"))
                except:
                    pass # نتجاهله إذا كان موجوداً

# إعداد المستخدم الوهمي إذا نجح بناء الأقسام
user_id = str(uuid.uuid4()) 
national_id = str(random.randint(10000000000000, 99999999999999))
username = f"testuser_{random.randint(1000,9999)}"

new_user = pd.DataFrame([{
    "Id": user_id, "FullName": "Test User", "NationalId": national_id, "IdCardImage": "dummy_image.jpg",
    "AccountStatus": 1, "Balance": 0.0, "CreatedAt": datetime.now(), "UserName": username,
    "NormalizedUserName": username.upper(), "Email": f"{username}@test.com", "NormalizedEmail": f"{username.upper()}@TEST.COM",
    "EmailConfirmed": False, "PhoneNumberConfirmed": False, "TwoFactorEnabled": False, "LockoutEnabled": False, "AccessFailedCount": 0
}])

try:
    print("جاري إنشاء المستخدم الوهمي...")
    new_user.to_sql('AspNetUsers', con=engine, if_exists='append', index=False)
    
    # ربط المنتجات
    print("جاري رفع المنتجات...")
    raw_data = [
        [user_id,1,101,"Nasr City",1,"Electronics","Sony",1,"PlayStation 5", "PS5 with 2 controllers",200,"Return in same condition",1,datetime.now()],
        [user_id,1,102,"Heliopolis",2,"Electronics","Microsoft",1,"Xbox Series X", "Xbox with wireless controller",180,"No damage allowed",1,datetime.now()],
        [user_id,2,201,"Maadi",1,"Sports","Trinx",0,"Mountain Bike", "Professional mountain bike",90,"Helmet included",1,datetime.now()],
        [user_id,2,202,"Dokki",2,"Sports","Galaxy",0,"City Bike", "Comfortable city bicycle",70,"Return clean",1,datetime.now()],
        [user_id,3,301,"Nasr City",1,"Electronics","Canon",1,"Canon DSLR Camera", "Canon DSLR with lens",150,"No scratches",1,datetime.now()],
        [user_id,3,302,"Helwan",2,"Electronics","Nikon",1,"Nikon Camera", "Nikon camera with tripod",140,"Handle carefully",1,datetime.now()],
        [user_id,4,401,"Zamalek",1,"Computers","Dell",1,"Dell Laptop", "Core i7 laptop 16GB RAM",220,"No software install",1,datetime.now()],
        [user_id,4,402,"Nasr City",2,"Computers","HP",0,"HP Laptop", "Core i5 laptop",150,"Return charged",1,datetime.now()],
        [user_id,5,501,"Maadi",1,"Projectors","Epson",1,"Epson Projector", "Full HD projector",130,"Indoor use only",1,datetime.now()],
        [user_id,5,502,"Dokki",2,"Projectors","BenQ",1,"BenQ Projector", "Portable projector",120,"No dropping",1,datetime.now()],
        [user_id,6,601,"Heliopolis",1,"Tools","Bosch",0,"Electric Drill", "Bosch heavy-duty drill",60,"Return with case",1,datetime.now()],
        [user_id,6,602,"Nasr City",2,"Tools","Makita",0,"Angle Grinder", "Makita grinder",55,"Safety required",1,datetime.now()],
        [user_id,7,701,"Maadi",1,"Gaming","Logitech",0,"Gaming Steering Wheel", "Wheel for racing games",80,"Return intact",1,datetime.now()],
        [user_id,7,702,"Dokki",2,"Gaming","Razer",0,"Gaming Keyboard", "RGB mechanical keyboard",40,"No liquid damage",1,datetime.now()],
        [user_id,8,801,"Zamalek",1,"Audio","JBL",0,"JBL Speaker", "Portable Bluetooth speaker",50,"Return charged",1,datetime.now()],
        [user_id,8,802,"Nasr City",2,"Audio","Sony",0,"Sony Headphones", "Noise cancelling headphones",45,"No scratches",1,datetime.now()],
        [user_id,9,901,"Heliopolis",1,"Photography","GoPro",1,"GoPro Hero 11", "Action camera waterproof",160,"Return accessories",1,datetime.now()],
        [user_id,9,902,"Maadi",2,"Photography","DJI",1,"DJI Gimbal", "Camera stabilizer",100,"Handle carefully",1,datetime.now()],
        [user_id,10,1001,"Nasr City",1,"Furniture","IKEA",0,"Office Chair", "Comfortable office chair",30,"Indoor use only",1,datetime.now()],
        [user_id,10,1002,"Dokki",2,"Furniture","Generic",0,"Folding Table", "Portable folding table",25,"Return clean",1,datetime.now()]
    ]

    products_data = pd.DataFrame(raw_data, columns=[
        "UserId","CategoryId","SubcategoryId","LocationArea","Condition",
        "ProductType","Brand","RentalGuarantee","Name","Description",
        "PricePerDay","TermsConditions","Status","CreatedAt"
    ])

    products_data.to_sql('Products', con=engine, if_exists='append', index=False)
    print("تم إضافة الأقسام، المستخدم، والـ 20 منتج بنجاح ✅ مبروووك!")

except Exception as e:
    print(f"حدث خطأ أخير: {e}")


SyntaxError: expected 'except' or 'finally' block (474165308.py, line 43)

In [16]:
import pandas as pd
import uuid
import random
from datetime import datetime
from sqlalchemy import text

# ==========================================
# 1. إعداد الأقسام والأقسام الفرعية
# ==========================================
try:
    with engine.begin() as conn:
        categories = [(1, 'Electronics'), (2, 'Sports'), (3, 'Cameras'), (4, 'Computers'), 
                      (5, 'Projectors'), (6, 'Tools'), (7, 'Gaming'), (8, 'Audio'), 
                      (9, 'Photography'), (10, 'Furniture')]
        
        print("جاري إنشاء الأقسام (Categories)...")
        for cid, cname in categories:
            try:
                conn.execute(text(f"SET IDENTITY_INSERT Categories ON; INSERT INTO Categories (Id, Name, CreatedAt) VALUES ({cid}, '{cname}', GETDATE()); SET IDENTITY_INSERT Categories OFF;"))
            except:
                try:
                    conn.execute(text(f"INSERT INTO Categories (Id, Name, CreatedAt) VALUES ({cid}, '{cname}', GETDATE())"))
                except: pass

        subcategories = [(101, 1, 'PlayStation'), (102, 1, 'Xbox'), (201, 2, 'Bikes'), (202, 2, 'City Bikes'),
                         (301, 3, 'DSLR'), (302, 3, 'Other'), (401, 4, 'Laptops'), (402, 4, 'PCs'),
                         (501, 5, 'HD Projectors'), (502, 5, 'Portable Projectors'), (601, 6, 'Drills'), (602, 6, 'Grinders'),
                         (701, 7, 'Steering Wheels'), (702, 7, 'Keyboards'), (801, 8, 'Speakers'), (802, 8, 'Headphones'),
                         (901, 9, 'Action Cam'), (902, 9, 'Gimbals'), (1001, 10, 'Chairs'), (1002, 10, 'Tables')]
                         
        print("جاري إنشاء الأقسام الفرعية (SubCategories)...")
        for sid, cid, sname in subcategories:
            try:
                conn.execute(text(f"SET IDENTITY_INSERT SubCategories ON; INSERT INTO SubCategories (Id, CategoryId, Name) VALUES ({sid}, {cid}, '{sname}'); SET IDENTITY_INSERT SubCategories OFF;"))
            except:
                try:
                    conn.execute(text(f"INSERT INTO SubCategories (Id, CategoryId, Name) VALUES ({sid}, {cid}, '{sname}')"))
                except: pass
                
except Exception as e:
    print(f"تنبيه (موقتاً): {e}")

# ==========================================
# 2. إعداد المستخدم ورفع المنتجات
# ==========================================
user_id = str(uuid.uuid4()) 
national_id = str(random.randint(10000000000000, 99999999999999))
username = f"testuser_{random.randint(1000,9999)}"

new_user = pd.DataFrame([{
    "Id": user_id, "FullName": "Test User", "NationalId": national_id, "IdCardImage": "dummy_image.jpg",
    "AccountStatus": 1, "Balance": 0.0, "CreatedAt": datetime.now(), "UserName": username,
    "NormalizedUserName": username.upper(), "Email": f"{username}@test.com", "NormalizedEmail": f"{username.upper()}@TEST.COM",
    "EmailConfirmed": False, "PhoneNumberConfirmed": False, "TwoFactorEnabled": False, "LockoutEnabled": False, "AccessFailedCount": 0
}])

try:
    print("جاري إنشاء المستخدم الوهمي...")
    new_user.to_sql('AspNetUsers', con=engine, if_exists='append', index=False)
    
    print("جاري رفع المنتجات...")
    raw_data = [
        [user_id,1,101,"Nasr City",1,"Electronics","Sony",1,"PlayStation 5", "PS5 with 2 controllers",200,"Return in same condition",1,datetime.now()],
        [user_id,1,102,"Heliopolis",2,"Electronics","Microsoft",1,"Xbox Series X", "Xbox with wireless controller",180,"No damage allowed",1,datetime.now()],
        [user_id,2,201,"Maadi",1,"Sports","Trinx",0,"Mountain Bike", "Professional mountain bike",90,"Helmet included",1,datetime.now()],
        [user_id,2,202,"Dokki",2,"Sports","Galaxy",0,"City Bike", "Comfortable city bicycle",70,"Return clean",1,datetime.now()],
        [user_id,3,301,"Nasr City",1,"Electronics","Canon",1,"Canon DSLR Camera", "Canon DSLR with lens",150,"No scratches",1,datetime.now()],
        [user_id,3,302,"Helwan",2,"Electronics","Nikon",1,"Nikon Camera", "Nikon camera with tripod",140,"Handle carefully",1,datetime.now()],
        [user_id,4,401,"Zamalek",1,"Computers","Dell",1,"Dell Laptop", "Core i7 laptop 16GB RAM",220,"No software install",1,datetime.now()],
        [user_id,4,402,"Nasr City",2,"Computers","HP",0,"HP Laptop", "Core i5 laptop",150,"Return charged",1,datetime.now()],
        [user_id,5,501,"Maadi",1,"Projectors","Epson",1,"Epson Projector", "Full HD projector",130,"Indoor use only",1,datetime.now()],
        [user_id,5,502,"Dokki",2,"Projectors","BenQ",1,"BenQ Projector", "Portable projector",120,"No dropping",1,datetime.now()],
        [user_id,6,601,"Heliopolis",1,"Tools","Bosch",0,"Electric Drill", "Bosch heavy-duty drill",60,"Return with case",1,datetime.now()],
        [user_id,6,602,"Nasr City",2,"Tools","Makita",0,"Angle Grinder", "Makita grinder",55,"Safety required",1,datetime.now()],
        [user_id,7,701,"Maadi",1,"Gaming","Logitech",0,"Gaming Steering Wheel", "Wheel for racing games",80,"Return intact",1,datetime.now()],
        [user_id,7,702,"Dokki",2,"Gaming","Razer",0,"Gaming Keyboard", "RGB mechanical keyboard",40,"No liquid damage",1,datetime.now()],
        [user_id,8,801,"Zamalek",1,"Audio","JBL",0,"JBL Speaker", "Portable Bluetooth speaker",50,"Return charged",1,datetime.now()],
        [user_id,8,802,"Nasr City",2,"Audio","Sony",0,"Sony Headphones", "Noise cancelling headphones",45,"No scratches",1,datetime.now()],
        [user_id,9,901,"Heliopolis",1,"Photography","GoPro",1,"GoPro Hero 11", "Action camera waterproof",160,"Return accessories",1,datetime.now()],
        [user_id,9,902,"Maadi",2,"Photography","DJI",1,"DJI Gimbal", "Camera stabilizer",100,"Handle carefully",1,datetime.now()],
        [user_id,10,1001,"Nasr City",1,"Furniture","IKEA",0,"Office Chair", "Comfortable office chair",30,"Indoor use only",1,datetime.now()],
        [user_id,10,1002,"Dokki",2,"Furniture","Generic",0,"Folding Table", "Portable folding table",25,"Return clean",1,datetime.now()]
    ]

    products_data = pd.DataFrame(raw_data, columns=[
        "UserId","CategoryId","SubcategoryId","LocationArea","Condition",
        "ProductType","Brand","RentalGuarantee","Name","Description",
        "PricePerDay","TermsConditions","Status","CreatedAt"
    ])

    products_data.to_sql('Products', con=engine, if_exists='append', index=False)
    print("✅ تمت إضافة الأقسام، والمستخدم الوهمي، والـ 20 منتج بنجاح!")

except Exception as e:
    print(f"❌ حدث خطأ أخير في إنشاء المستخدم أو الرفع: {e}")


جاري إنشاء الأقسام (Categories)...
جاري إنشاء الأقسام الفرعية (SubCategories)...
جاري إنشاء المستخدم الوهمي...
جاري رفع المنتجات...
❌ حدث خطأ أخير في إنشاء المستخدم أو الرفع: (pyodbc.ProgrammingError) ('42S22', "[42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'PricePerDay'. (207) (SQLExecDirectW); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'PricePerDay'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")
[SQL: INSERT INTO [Products] ([UserId], [CategoryId], [SubcategoryId], [LocationArea], [Condition], [ProductType], [Brand], [RentalGuarantee], [Name], [Description], [PricePerDay], [TermsConditions], [Status], [CreatedAt]) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ? ... 752 characters truncated ...  ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('1b7ef8d7-8e5e-4174-89ff-f1a63402b499', 1, 101, 'N

In [1]:
import pandas as pd

try:
    # كتابة الاستعلام لجلب كل البيانات من جدول Products
    query = "SELECT * FROM Products"
    
    # قراءة البيانات وعرضها في DataFrame
    df_products = pd.read_sql_query(query, engine)
    
    # عرض البيانات (ستظهر كجدول منسق)
    display(df_products)
    
    # إذا كان الجدول كبيراً جداً وتريد عرض أول 10 صفوف فقط، يمكنك استخدام:
    # display(df_products.head(10))
    
except Exception as e:
    print(f"Failed to fetch data: {e}")

Failed to fetch data: name 'engine' is not defined


In [7]:
import pandas as pd

query = """
SELECT 
    p.Id,
    
    -- بيانات المستخدم
    p.UserId,
    u.FullName AS UserName,  -- أو UserName بدل FullName حسب المطلوب المعروض
    
    -- بيانات القسم
    p.CategoryId,
    c.Name AS CategoryName,
    
    -- بيانات القسم الفرعي
    p.SubcategoryId,
    s.Name AS SubcategoryName,
    
    -- باقي بيانات المنتج
    p.LocationArea, 
    p.Condition, 
    p.ProductType, 
    p.Brand, 
    p.RentalGuarantee, 
    p.Name, 
    p.Description, 
    p.PricePerDay, 
    p.TermsConditions, 
    p.Status, 
    p.CreatedAt
    
FROM Products p
LEFT JOIN AspNetUsers u ON p.UserId = u.Id
LEFT JOIN Categories c ON p.CategoryId = c.Id
LEFT JOIN SubCategories s ON p.SubcategoryId = s.Id
"""

try:
    # جلب البيانات المرتبطة من قاعدة البيانات
    df_products_with_names = pd.read_sql_query(query, engine)
    
    # عرض الجدول الجميل والمقروء
    display(df_products_with_names)
    
except Exception as e:
    print(f"حدث خطأ أثناء جلب البيانات: {e}")


حدث خطأ أثناء جلب البيانات: (pyodbc.ProgrammingError) ('42S22', "[42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'PricePerDay'. (207) (SQLExecDirectW)")
[SQL: 
SELECT 
    p.Id,
    
    -- بيانات المستخدم
    p.UserId,
    u.FullName AS UserName,  -- أو UserName بدل FullName حسب المطلوب المعروض
    
    -- بيانات القسم
    p.CategoryId,
    c.Name AS CategoryName,
    
    -- بيانات القسم الفرعي
    p.SubcategoryId,
    s.Name AS SubcategoryName,
    
    -- باقي بيانات المنتج
    p.LocationArea, 
    p.Condition, 
    p.ProductType, 
    p.Brand, 
    p.RentalGuarantee, 
    p.Name, 
    p.Description, 
    p.PricePerDay, 
    p.TermsConditions, 
    p.Status, 
    p.CreatedAt
    
FROM Products p
LEFT JOIN AspNetUsers u ON p.UserId = u.Id
LEFT JOIN Categories c ON p.CategoryId = c.Id
LEFT JOIN SubCategories s ON p.SubcategoryId = s.Id
]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [6]:
from sqlalchemy import text

# إحنا بنقسم الأكواد لأن الـ SQL بيطلب إن أمر Create View يتنفذ لوحده
drop_old_table = """
-- مسح الجدول الثابت لو كان موجود
IF OBJECT_ID('Products_LLm', 'U') IS NOT NULL 
    DROP TABLE Products_LLm;
"""

drop_old_view = """
-- مسح الـ View لو كان موجود قبل كده
IF OBJECT_ID('Products_LLm', 'V') IS NOT NULL 
    DROP VIEW Products_LLm;
"""

create_view_query = """
-- إنشاء الـ View الحي اللي بيربط البيانات ويحدثها تلقائياً
CREATE VIEW Products_LLm AS
SELECT 
    p.Id,
    p.UserId,
    u.FullName AS UserName,
    p.CategoryId,
    c.Name AS CategoryName,
    p.SubcategoryId,
    s.Name AS SubcategoryName,
    p.LocationArea, 
    p.Condition, 
    p.ProductType, 
    p.Brand, 
    p.RentalGuarantee, 
    p.Name, 
    p.Description, 
    p.PricePerDay, 
    p.TermsConditions, 
    p.Status, 
    p.CreatedAt
FROM Products p
LEFT JOIN AspNetUsers u ON p.UserId = u.Id
LEFT JOIN Categories c ON p.CategoryId = c.Id
LEFT JOIN SubCategories s ON p.SubcategoryId = s.Id;
"""

try:
    with engine.begin() as conn:
        conn.execute(text(drop_old_table))
        conn.execute(text(drop_old_view))
        conn.execute(text(create_view_query))
        
        print("تم مسح الجدول الثابت، وإنشاء المنظور الحي (VIEW) باسم 'Products_LLm' بنجاح! 🚀")
        print("دلوقتي أي منتج هيتطاف في الجداول الأصلية، هيسمّع هنا فوراً وفي نفس اللحظة.")
        
except Exception as e:
    print(f"حدث خطأ أثناء الإنشاء: {e}")


حدث خطأ أثناء الإنشاء: (pyodbc.ProgrammingError) ('42S22', "[42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'PricePerDay'. (207) (SQLExecDirectW)")
[SQL: 
-- إنشاء الـ View الحي اللي بيربط البيانات ويحدثها تلقائياً
CREATE VIEW Products_LLm AS
SELECT 
    p.Id,
    p.UserId,
    u.FullName AS UserName,
    p.CategoryId,
    c.Name AS CategoryName,
    p.SubcategoryId,
    s.Name AS SubcategoryName,
    p.LocationArea, 
    p.Condition, 
    p.ProductType, 
    p.Brand, 
    p.RentalGuarantee, 
    p.Name, 
    p.Description, 
    p.PricePerDay, 
    p.TermsConditions, 
    p.Status, 
    p.CreatedAt
FROM Products p
LEFT JOIN AspNetUsers u ON p.UserId = u.Id
LEFT JOIN Categories c ON p.CategoryId = c.Id
LEFT JOIN SubCategories s ON p.SubcategoryId = s.Id;
]
(Background on this error at: https://sqlalche.me/e/20/f405)
